# ============================================================
# T5 Fine-Tuning — Medical Jargon Simplification
# Platform  : Kaggle (T4 GPU) / Google Colab
# Input     : medical_simplified_final.csv  (your uploaded CSV)
# Output    : /kaggle/working/t5-medical-finetuned/
# ============================================================



# ════════════════════════════════════════════════════════════
# CELL 1 — Install Dependencies
# ════════════════════════════════════════════════════════════


In [1]:
!pip install -q transformers==4.40.0 flax jax

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.


In [2]:
!pip install transformers datasets accelerate sentencepiece rouge_score scikit-learn -q print("All dependencies installed.")

/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `pip install transformers datasets accelerate sentencepiece rouge_score scikit-learn -q print("All dependencies installed.")'


In [3]:
!pip install -q \
    "transformers==4.41.0" \
    "peft==0.11.0" \
    "accelerate==0.30.0" \
    datasets sentencepiece rouge_score scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 22.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.2/251.2 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.4/302.4 kB 22.0 MB/s eta 0:00:00


# CELL 2 — Imports & GPU Check

In [4]:
import os
import gc
import json
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from rouge_score import rouge_scorer
from datasets import Dataset as HFDataset
from transformers import (
    AutoTokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)

os.environ['WANDB_DISABLED']         = 'true'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU — Settings → Accelerator → GPU T4 x2')

print('All imports done.')

2026-03-05 09:50:31.247334: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772704231.447640     109 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772704231.506305     109 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772704231.951961     109 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772704231.952021     109 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772704231.952024     109 computation_placer.cc:177] computation placer alr

Device : cuda
GPU    : Tesla P100-PCIE-16GB
VRAM   : 17.1 GB
All imports done.


# CELL 3 — Load Your Uploaded CSV

In [5]:
# First find exact path of your uploaded file
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/smritiiiiii60/medicaldatas/medical_simplified_final.csv


In [6]:
# UPDATE path from output above
df = pd.read_csv('/kaggle/input/datasets/smritiiiiii60/medicaldatas/medical_simplified_final.csv')
df.head()

,medical,simple
0,"Under conditions of high humidity , the rate o...","With a higher humidity , the rate of evaporati..."
1,"The lack of oxygen above 2,400 metres ( 8,000 ...",This can cause illnesses such as altitude sick...
2,The human body can adapt to high altitude by b...,The human body can deal with high altitude by ...
3,"For example , hemoglobin and myoglobin contain...","For example , hemoglobin and myoglobin contain..."
4,"Schistosomiasis , caused by one genus of trema...","Schistosomiasis , caused by one genus of trema..."


In [7]:
df.isnull().sum()

medical       0
simple     3426
dtype: int64

In [8]:
df.shape

(15738, 2)

In [9]:
# ── Adapt column names if yours differ ──────────────────────
# Expected: 'medical' (input) and 'simple' (target)
# If your CSV has different column names, rename them here:
#   df = df.rename(columns={'your_medical_col': 'medical', 'your_simple_col': 'simple'})

print('Columns available:', list(df.columns))

# Auto-detect columns if not named 'medical'/'simple'
if 'medical' not in df.columns or 'simple' not in df.columns:
    cols = list(df.columns)
    print(f'Renaming: {cols[0]} → medical, {cols[1]} → simple')
    df = df.rename(columns={cols[0]: 'medical', cols[1]: 'simple'})

# Drop nulls and duplicates
df = df[['medical', 'simple']].dropna().drop_duplicates().reset_index(drop=True)
df['medical'] = df['medical'].astype(str).str.strip()
df['simple']  = df['simple'].astype(str).str.strip()
# Remove empty rows
df = df[(df['medical'] != '') & (df['simple'] != '')].reset_index(drop=True)

print(f'Clean rows: {len(df):,}')

# Train / Val split  (90 / 10)
train_df, val_df = train_test_split(df, test_size=0.10, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
print(f'Train : {len(train_df):,}  |  Val : {len(val_df):,}')
print(train_df.head(3))

Columns available: ['medical', 'simple']
Clean rows: 12,312
Train : 11,080  |  Val : 1,232
                                             medical  \
0  Five studies were included, of which four with...   
1  The anti-VSC (volatile sulphur compounds) effe...   
2  On Mother 's Day 1944, while on leave during W...   

                                              simple  
0  This review found five poor to moderate qualit...  
1  Zinc is known to interfere with volatile sulph...  
2  On Mothers ' Day in 1944, his mother committed...  


### Load ClinicalT5-base Model & Tokenizer


In [10]:
# Clear old model if re-running
try:
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print('Old model cleared.')
except Exception:
    pass

model_name = 'google/flan-t5-base'

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(model_name)

print('Loading model in float32...')
model = T5ForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float32,   # float32 for stability
    tie_word_embeddings=True,    # ✅ FIX 1: ties encoder/decoder embed weights
                                 #    → fixes "missing keys" warning & loss=0.0 bug
).to(device)

# ✅ FIX 1b: Explicitly call tie_weights() to guarantee weight sharing
model.tie_weights()

print(f'Model loaded on  : {device}')
print(f'Parameters       : {sum(p.numel() for p in model.parameters())/1e6:.0f}M')
if device == 'cuda':
    print(f'VRAM used        : {torch.cuda.memory_allocated(0)/1e9:.1f} GB')


Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Loading model in float32...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded on  : cuda
Parameters       : 223M
VRAM used        : 0.9 GB


### Train / Validation Split

### Tokenise
Using prefix `simplify medical text:` — tells T5 exactly what task to perform

In [11]:
# Clear old model if re-running
try:
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print('Old model cleared.')
except Exception:
    pass

model_name = 'google/flan-t5-base'

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(model_name)

print('Loading model...')
model = T5ForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    ignore_mismatched_sizes=False,
)

# ✅ KEY FIX: Manually tie all 3 weights explicitly
# This forces encoder, decoder, and lm_head to share the same embedding matrix
model.encoder.embed_tokens.weight = model.shared.weight
model.decoder.embed_tokens.weight = model.shared.weight
model.lm_head.weight              = model.shared.weight

# Verify no missing weights
print('✅ Embedding weights manually tied')
print(f'shared weight shape      : {model.shared.weight.shape}')
print(f'encoder embed shape      : {model.encoder.embed_tokens.weight.shape}')
print(f'decoder embed shape      : {model.decoder.embed_tokens.weight.shape}')
print(f'lm_head shape            : {model.lm_head.weight.shape}')

model = model.to(device)
print(f'Model loaded on  : {device}')
print(f'Parameters       : {sum(p.numel() for p in model.parameters())/1e6:.0f}M')
if device == 'cuda':
    print(f'VRAM used        : {torch.cuda.memory_allocated(0)/1e9:.1f} GB')


Old model cleared.
Loading tokenizer...
Loading model...
✅ Embedding weights manually tied
shared weight shape      : torch.Size([32128, 768])
encoder embed shape      : torch.Size([32128, 768])
decoder embed shape      : torch.Size([32128, 768])
lm_head shape            : torch.Size([32128, 768])
Model loaded on  : cuda
Parameters       : 223M
VRAM used        : 0.9 GB


In [12]:
model = T5ForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    tie_word_embeddings=True,
)

model.encoder.embed_tokens.weight = model.shared.weight
model.decoder.embed_tokens.weight = model.shared.weight
model.lm_head.weight              = model.shared.weight

# ✅ FIX OOM: enable gradient checkpointing on model itself
model.gradient_checkpointing_enable()

model = model.to(device)
torch.cuda.empty_cache()
print(f'Model loaded | VRAM: {torch.cuda.memory_allocated(0)/1e9:.1f} GB')

Model loaded | VRAM: 0.9 GB


In [13]:
PREFIX     = 'simplify medical text: '
MAX_INPUT  = 256
MAX_TARGET = 128

def make_hf_dataset(df, tokenizer, prefix, max_input, max_target):
    def tokenise(batch):
        # ✅ FIX 2: Do NOT use padding='max_length' here.
        #    Let DataCollatorForSeq2Seq handle dynamic padding per batch.
        #    Static max_length padding inflates attention over pad tokens → NaN loss.
        model_inputs = tokenizer(
            [prefix + t for t in batch['medical']],
            max_length=max_input,
            truncation=True,
            padding=False,       # ✅ no pre-padding; collator will pad dynamically
        )

        labels = tokenizer(
            text_target=batch['simple'],
            max_length=max_target,
            truncation=True,
            padding=False,       # ✅ same — let the collator handle padding
        )

        # Replace pad token id with -100 so loss ignores padding
        model_inputs['labels'] = [
            [(l if l != tokenizer.pad_token_id else -100) for l in label]
            for label in labels['input_ids']
        ]
        return model_inputs

    hf = HFDataset.from_pandas(df[['medical', 'simple']])
    hf = hf.map(
        tokenise,
        batched=True,
        batch_size=64,
        remove_columns=['medical', 'simple'],
    )
    return hf

print('Tokenising train set...')
train_dataset = make_hf_dataset(train_df, tokenizer, PREFIX, MAX_INPUT, MAX_TARGET)
print('Tokenising val set...')
val_dataset   = make_hf_dataset(val_df,   tokenizer, PREFIX, MAX_INPUT, MAX_TARGET)

print(f'\nTrain : {len(train_dataset):,}  |  Val : {len(val_dataset):,}')
print(train_dataset)

# Sanity check
sample   = train_dataset[0]
non_pad  = [l for l in sample['labels'] if l != -100]
print(f'\nSanity check sample[0]:')
print(f'  input_ids length  : {len(sample["input_ids"])}')
print(f'  labels length     : {len(sample["labels"])}')
print(f'  Non-padding tokens: {len(non_pad)}')
print(f'  Decoded label     : {tokenizer.decode(non_pad, skip_special_tokens=True)}')
if len(non_pad) > 0:
    print('✅ Tokenization OK — labels are set correctly.')
else:
    print('❌ WARNING: All labels are -100! Check your CSV column names.')


Tokenising train set...


Map:   0%|          | 0/11080 [00:00<?, ? examples/s]

Tokenising val set...


Map:   0%|          | 0/1232 [00:00<?, ? examples/s]


Train : 11,080  |  Val : 1,232
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 11080
})

Sanity check sample[0]:
  input_ids length  : 243
  labels length     : 79
  Non-padding tokens: 79
  Decoded label     : This review found five poor to moderate quality studies, of which four with a total of 282 women provided data. There was not enough evidence to say if systematic desensitisation worked better than another treatment. Further studies including larger numbers of women are needed to show if systematic desensitisation if effective for the treatment of women with vaginismus.
✅ Tokenization OK — labels are set correctly.


### Cell 9 — ROUGE Metric

In [14]:
from rouge_score import rouge_scorer as rs_module

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # ✅ FIX: Convert logits → token IDs if needed (shape check)
    if hasattr(predictions, 'ndim') and predictions.ndim == 3:
        predictions = predictions.argmax(axis=-1)

    # Clip to valid vocab range to avoid OverflowError
    vocab_size   = tokenizer.vocab_size
    predictions  = np.clip(predictions, 0, vocab_size - 1).astype(np.int32)

    decoded_preds  = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 in labels (padding) with pad_token_id before decoding
    labels         = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels         = np.clip(labels, 0, vocab_size - 1).astype(np.int32)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Strip whitespace
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    # Compute ROUGE
    scorer  = rs_module.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    r1, r2, rl = [], [], []
    for pred, label in zip(decoded_preds, decoded_labels):
        scores = scorer.score(label, pred)
        r1.append(scores['rouge1'].fmeasure)
        r2.append(scores['rouge2'].fmeasure)
        rl.append(scores['rougeL'].fmeasure)

    return {
        'rouge1': round(np.mean(r1), 4),
        'rouge2': round(np.mean(r2), 4),
        'rougeL': round(np.mean(rl), 4),
    }

print('compute_metrics defined ✅')

compute_metrics defined ✅


### Training Arguments
Tuned for lowest validation loss on T4 GPU

In [15]:
OUTPUT_DIR = '/kaggle/working/t5-medical-finetuned'
# ✅ FIX 3: fp16 with T5 float32 weights causes NaN validation loss.
#    Use bf16 if the GPU supports it (Ampere+), otherwise train in float32.
use_fp16 = False
use_bf16 = (device == 'cuda' and torch.cuda.is_bf16_supported())
training_args = Seq2SeqTrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = 15,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    warmup_steps                = 500,            # slightly more warmup for stability
    weight_decay                = 0.001,
    learning_rate               = 5e-4,
    lr_scheduler_type           = 'cosine',
    # ✅ predict_with_generate → eval uses model.generate() → real token IDs
    predict_with_generate       = True,
    generation_max_length       = MAX_TARGET,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'rougeL',
    greater_is_better           = True,
    logging_dir                 = f'{OUTPUT_DIR}/logs',
    logging_steps               = 50,
    save_total_limit            = 2,
    fp16                        = use_fp16,       # ✅ FIX 3a: disabled (caused NaN val loss)
    bf16                        = use_bf16,       # ✅ FIX 3b: safe alternative if available
    label_smoothing_factor      = 0.1,            # ✅ FIX 3c: prevents loss collapsing to 0
    report_to                   = 'none',
    dataloader_num_workers      = 4,
)
# ✅ FIX 3d: padding=True → dynamic padding per batch (NOT max_length)
#    Consistent with tokenisation step — avoids NaN from over-padded attention
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if device == 'cuda' else None,
    padding=True,                                 # ✅ dynamic padding
)
trainer = Seq2SeqTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    tokenizer       = tokenizer,
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
)
print('Trainer ready ✅')
print(f'fp16 = {use_fp16}  |  bf16 = {use_bf16}')
print(f'Train batches/epoch : {len(train_dataset) // training_args.per_device_train_batch_size}')

Trainer ready ✅
fp16 = False  |  bf16 = True
Train batches/epoch : 692


## Trainer

### Train
Watch `eval_loss` drop each 500 steps. Target: **< 0.47**

In [16]:
# ✅ Clear GPU cache before training starts
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
torch.cuda.empty_cache()
gc.collect()





190

In [17]:
print('─' * 65)
print('Starting training...')
print('─' * 65)

train_result = trainer.train()

print('─' * 65)
print('Training complete!')
print(f"  Train Loss : {train_result.training_loss:.4f}")
print(f"  Total Steps: {train_result.global_step}")
print(f"  Runtime    : {train_result.metrics.get('train_runtime', 0)/60:.1f} min")
print('─' * 65)

─────────────────────────────────────────────────────────────────
Starting training...
─────────────────────────────────────────────────────────────────


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
1,6.410500,6.204428,0.052100,0.004100,0.048700
2,5.723200,5.581469,0.068400,0.005500,0.056400
3,5.334700,5.201243,0.137000,0.014500,0.106600
4,4.985100,4.867748,0.121200,0.012300,0.095000
5,4.722000,4.630673,0.145600,0.015700,0.113400
6,4.533100,4.475196,0.163400,0.018400,0.128100
7,4.357100,4.348386,0.172100,0.021800,0.135600
8,4.218800,4.249135,0.170600,0.025400,0.136400
9,4.082500,4.142280,0.215300,0.035200,0.168500
10,3.936900,3.967014,0.258800,0.065900,0.215500


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


─────────────────────────────────────────────────────────────────
Training complete!
  Train Loss : 4.5541
  Total Steps: 10395
  Runtime    : 316.4 min
─────────────────────────────────────────────────────────────────


### Cell 12 — Evaluate

In [18]:
metrics = trainer.evaluate()

print('\n═══ Final Evaluation Results ══════════════')
print(f"  ROUGE-1 : {metrics.get('eval_rouge1', 'N/A')}")
print(f"  ROUGE-2 : {metrics.get('eval_rouge2', 'N/A')}")
print(f"  ROUGE-L : {metrics.get('eval_rougeL', 'N/A')}")
print(f"  Val Loss: {metrics.get('eval_loss',   'N/A')}")
print('════════════════════════════════════════════')

vl = metrics.get('eval_loss', 99)
if   vl < 0.47: print('✅ Excellent! Val loss < 0.47')
elif vl < 0.55: print('✅ Good. Val loss < 0.55')
else:           print('⚠️  Val loss > 0.55 — try more epochs.')


═══ Final Evaluation Results ══════════════
  ROUGE-1 : 0.3713
  ROUGE-2 : 0.1619
  ROUGE-L : 0.3245
  Val Loss: 3.643950939178467
════════════════════════════════════════════
⚠️  Val loss > 0.55 — try more epochs.


### Cell 13 — Save Model

In [19]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(f'{OUTPUT_DIR}/eval_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

with open(f'{OUTPUT_DIR}/training_config.json', 'w') as f:
    json.dump({
        'base_model':    model_name,
        'prefix':        PREFIX,
        'max_input':     MAX_INPUT,
        'max_target':    MAX_TARGET,
        'train_samples': len(train_dataset),
        'val_samples':   len(val_dataset),
        'val_loss':      metrics.get('eval_loss'),
        'rougeL':        metrics.get('eval_rougeL'),
    }, f, indent=2)

print(f'Model saved to : {OUTPUT_DIR}')
print('\nFiles:')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = f'{OUTPUT_DIR}/{fname}'
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1024**2
        print(f'  {fname:<40} {size:.1f} MB')

Model saved to : /kaggle/working/t5-medical-finetuned

Files:
  config.json                              0.0 MB
  eval_metrics.json                        0.0 MB
  generation_config.json                   0.0 MB
  model.safetensors                        850.3 MB
  special_tokens_map.json                  0.0 MB
  spiece.model                             0.8 MB
  tokenizer.json                           2.3 MB
  tokenizer_config.json                    0.0 MB
  training_args.bin                        0.0 MB
  training_config.json                     0.0 MB


### Cell 14 — Test Inference

In [20]:
model.eval()

def simplify(text, max_length=128):
    input_ids = tokenizer(
        PREFIX + text,
        return_tensors='pt',
        max_length=MAX_INPUT,
        truncation=True,
    ).input_ids.to(device)

    with torch.no_grad():
        out = model.generate(
            input_ids,
            max_length=max_length,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

tests = [
    'The patient presents with acute myocardial infarction.',
    'Hypertension requires antihypertensive pharmacotherapy.',
    'Bilateral pneumonia with pleural effusion was confirmed.',
    'The lack of oxygen above 2,400 metres can cause altitude sickness and high altitude cerebral edema.',
    'The patient has Type 2 diabetes mellitus with peripheral neuropathy.',
]

print('═══ Inference Tests ════════════════════════')
for text in tests:
    print(f'\n  Input : {text}')
    print(f'  Output: {simplify(text)}')
print('════════════════════════════════════════════')

═══ Inference Tests ════════════════════════

  Input : The patient presents with acute myocardial infarction.
  Output: The patient suffered with severe haemorrhage in the chest.

  Input : Hypertension requires antihypertensive pharmacotherapy.
  Output: Hypertension can be caused by antidepressant drugs.

  Input : Bilateral pneumonia with pleural effusion was confirmed.
  Output: Covid-19 with bronchitis was reported.

  Input : The lack of oxygen above 2,400 metres can cause altitude sickness and high altitude cerebral edema.
  Output: The lack of oxygen in the blood can cause hypertension and high blood pressure.

  Input : The patient has Type 2 diabetes mellitus with peripheral neuropathy.
  Output: The patient has 2 diabetes mellitus with neuropathy.
════════════════════════════════════════════
